# 08 — Telugu ASR: Multi-Model Learning Curve Runner

**Research question:** What is the minimum viable training data size for practical Telugu ASR, and how does this trade-off differ across transformer architectures?

This notebook fully automates the experiment matrix. It shuffles all `(model × fraction × source)` combinations, skips any already-completed runs, trains sequentially, and redraws a live learning curve plot after every completed run.

| Model | Architecture | Fractions | Source |
|---|---|---|---|
| wav2vec2-xlsr | CTC | 0.10, 0.25, 0.50, 1.00 | mscil |
| whisper-small | seq2seq | 0.10, 0.25, 0.50, 1.00 | mscil |
| whisper-medium | seq2seq | 0.50, 1.00 | mscil |
| mms-300m | CTC adapter | 0.0 (zero-shot), 1.00 | mscil |
| whisper-telugu | seq2seq (continued FT) | 1.00 | multi |

**Evaluation benchmark:** FLEURS te_in test set (held-out — never used for training)

In [ ]:
!pip install -q \
    "typing_extensions>=4.6" \
    "transformers>=4.46" \
    "datasets<3.0" \
    "accelerate>=0.26" \
    "evaluate" \
    "jiwer" \
    "soundfile" \
    "librosa" \
    "pyctcdecode" \
    "kenlm" \
    "matplotlib" \
    "ipywidgets"
print("Install complete.")

In [ ]:
import os, gc, re, json, random, logging, subprocess, sys, datetime
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import numpy as np
import torch
import evaluate
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from datasets import load_from_disk, load_dataset, Audio
from transformers import (
    Wav2Vec2ForCTC, Wav2Vec2Processor,
    WhisperForConditionalGeneration, WhisperProcessor,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)

logging.getLogger("numba").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.WARNING)

# ── GPU check ─────────────────────────────────────────────────────
assert torch.cuda.is_available(), "No GPU found — cannot run training."
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
torch.backends.cuda.matmul.allow_tf32 = True
print(f"GPU: {torch.cuda.get_device_name(0)}  ({vram_gb:.0f} GB VRAM)")

# ── Global paths ───────────────────────────────────────────────────
RESULTS_DIR   = Path("./results")
RESULTS_DIR.mkdir(exist_ok=True)
KENLM_BIN     = "./kenlm/build/bin/lmplz"
VOCAB_JSON    = "./vocab_v2.json" if Path("./vocab_v2.json").exists() else "./vocab.json"
SEED          = 42

SOURCE_PATHS = {
    "mscil": "./telugu_asr_clean_dataset",
    "multi": "./telugu_multi_dataset",
}

# ── Model registry ────────────────────────────────────────────────
# Memory-adaptive batch sizing for RTX A6000 (51 GB)
# wav2vec2: batch=8, grad_accum=4  → effective 32
# whisper-small/MMS: batch=16, grad_accum=2  → effective 32
# whisper-medium: batch=8, grad_accum=4  → effective 32
# whisper-large: batch=4, grad_accum=8  → effective 32

MODEL_REGISTRY = {
    "wav2vec2-xlsr": {
        "model_id":    "facebook/wav2vec2-xls-r-300m",
        "arch":        "ctc",
        "precision":   "fp16",
        "lr":          1e-4,
        "warmup":      1000,
        "epochs":      50,
        "patience":    8,
        "threshold":   0.005,
        "batch":       8,
        "grad_accum":  4,
        "mask_time_prob":    0.1,
        "mask_feature_prob": 0.1,
        "attention_dropout": 0.1,
        "hidden_dropout":    0.1,
        "feat_proj_dropout": 0.0,
        "eval_steps":  500,
        "kenlm":       True,
    },
    "whisper-small": {
        "model_id":   "openai/whisper-small",
        "arch":       "seq2seq",
        "precision":  "bf16",
        "lr":         1e-5,
        "warmup":     500,
        "epochs":     30,
        "patience":   5,
        "threshold":  0.005,
        "batch":      16,
        "grad_accum": 2,
        "eval_steps": 500,
        "kenlm":      False,
    },
    "whisper-medium": {
        "model_id":   "openai/whisper-medium",
        "arch":       "seq2seq",
        "precision":  "bf16",
        "lr":         5e-6,
        "warmup":     500,
        "epochs":     20,
        "patience":   5,
        "threshold":  0.005,
        "batch":      8,
        "grad_accum": 4,
        "eval_steps": 500,
        "kenlm":      False,
    },
    "mms-300m": {
        "model_id":   "facebook/mms-300m",
        "arch":       "ctc",
        "precision":  "fp16",
        "lr":         1e-4,
        "warmup":     500,
        "epochs":     20,
        "patience":   5,
        "threshold":  0.005,
        "batch":      16,
        "grad_accum": 2,
        "eval_steps": 400,
        "kenlm":      False,
        "adapter":    True,
    },
    "whisper-telugu": {
        "model_id":   "vasista22/whisper-telugu-large-v2",
        "arch":       "seq2seq",
        "precision":  "bf16",
        "lr":         2e-6,
        "warmup":     200,
        "epochs":     10,
        "patience":   3,
        "threshold":  0.005,
        "batch":      4,
        "grad_accum": 8,
        "eval_steps": 200,
        "kenlm":      False,
    },
}

# ── Experiment matrix ─────────────────────────────────────────────
MODELS_TO_RUN = ["wav2vec2-xlsr", "whisper-small", "whisper-medium", "mms-300m", "whisper-telugu"]

FRACTIONS = {
    "wav2vec2-xlsr":  [0.10, 0.25, 0.50, 1.00],
    "whisper-small":  [0.10, 0.25, 0.50, 1.00],
    "whisper-medium": [0.50, 1.00],
    "mms-300m":       [0.0,  1.00],   # 0.0 = zero-shot
    "whisper-telugu": [1.00],
}

SOURCES = {
    "wav2vec2-xlsr":  "mscil",
    "whisper-small":  "mscil",
    "whisper-medium": "mscil",
    "mms-300m":       "mscil",
    "whisper-telugu": "multi",
}

# ── Print experiment plan ─────────────────────────────────────────
total = sum(len(FRACTIONS[m]) for m in MODELS_TO_RUN)
print(f"\nTotal experiments planned: {total}")
print(f"{'Model':<20} {'Frac':<6} {'Source':<8} {'Result file'}")
print("-" * 70)
for m in MODELS_TO_RUN:
    for frac in FRACTIONS[m]:
        src = SOURCES[m]
        tag = f"{frac:.2f}"
        rpath = RESULTS_DIR / f"{m}_lc_{src}_{tag}.json"
        status = "DONE" if rpath.exists() else "pending"
        print(f"{m:<20} {str(frac):<6} {src:<8} {rpath.name}  [{status}]")

In [ ]:
# ── Helper: Load & subsample dataset ──────────────────────────────

def load_split(source: str, fraction: float, seed: int):
    """Load, subsample, and split a dataset."""
    ds_path = SOURCE_PATHS[source]
    assert Path(ds_path).exists(), f"Dataset not found: {ds_path}"
    ds = load_from_disk(ds_path)
    ds = ds.cast_column("audio", Audio(sampling_rate=16_000))

    if fraction < 1.0:
        n_keep = int(len(ds) * fraction)
        ds = ds.shuffle(seed=seed).select(range(n_keep))

    split = ds.train_test_split(test_size=0.1, seed=seed)
    train_ds = split["train"]
    val_ds   = split["test"]

    # Approximate hours: mean ~3.85s per clip
    approx_hours = round(len(train_ds) * 3.85 / 3600, 1)
    print(f"  Source={source}, fraction={fraction}")
    print(f"  Train: {len(train_ds):,}  |  Val: {len(val_ds):,}  |  ~{approx_hours}h")
    return train_ds, val_ds, len(train_ds), approx_hours


print("load_split() defined.")

In [ ]:
# ── Helper: Load FLEURS te_in once (held-out benchmark) ───────────

fleurs_test = None
try:
    _fl = load_dataset("google/fleurs", "te_in", trust_remote_code=True)
    fleurs_test = _fl["test"].select_columns(["audio", "transcription"])
    fleurs_test = fleurs_test.cast_column("audio", Audio(sampling_rate=16_000))
    print(f"FLEURS te_in test: {len(fleurs_test):,} samples (primary benchmark)")
except Exception as e:
    print(f"[WARN] FLEURS te_in unavailable: {e}")
    print("       FLEURS WER columns will be NaN in results.")

In [ ]:
# ── Helper: Live learning curve plot ──────────────────────────────

MODEL_COLORS = {
    "wav2vec2-xlsr":  "#2196F3",
    "whisper-small":  "#4CAF50",
    "whisper-medium": "#FF9800",
    "mms-300m":       "#9C27B0",
    "whisper-telugu": "#F44336",
}
MODEL_MARKERS = {
    "wav2vec2-xlsr":  "o",
    "whisper-small":  "s",
    "whisper-medium": "^",
    "mms-300m":       "D",
    "whisper-telugu": "*",
}

def update_plot():
    """Read all result JSONs and redraw learning curve in-place."""
    try:
        all_results = []
        for p in sorted(RESULTS_DIR.glob("*_lc_*.json")):
            try:
                with open(p) as f:
                    all_results.append(json.load(f))
            except Exception:
                pass

        if not all_results:
            print("No results yet — plot skipped.")
            return

        fig, ax = plt.subplots(figsize=(12, 7))

        by_model: Dict[str, list] = {}
        for r in all_results:
            mk = r.get("model_key", "unknown")
            by_model.setdefault(mk, []).append(r)

        for mk, runs in by_model.items():
            color  = MODEL_COLORS.get(mk, "grey")
            marker = MODEL_MARKERS.get(mk, "o")

            # Primary WER: kenlm > beam > greedy (best available)
            pts_primary = []
            pts_greedy  = []
            for r in sorted(runs, key=lambda x: x.get("n_train_samples", 0)):
                n = r.get("n_train_samples", 0)
                if n == 0:
                    n = 1  # zero-shot: plot at x=1 for log scale

                # Primary (best) WER on FLEURS te_in
                fleurs_wer = (
                    r.get("kenlm_wer_fleurs")
                    or r.get("beam_wer_fleurs")
                    or r.get("greedy_wer_fleurs")
                )
                if fleurs_wer is not None and not (isinstance(fleurs_wer, float) and np.isnan(fleurs_wer)):
                    pts_primary.append((n, fleurs_wer * 100))

                # Greedy (dashed, for CTC models)
                gw = r.get("greedy_wer_fleurs") or r.get("greedy_wer_val")
                if gw is not None and not (isinstance(gw, float) and np.isnan(gw)):
                    pts_greedy.append((n, gw * 100))

            if pts_primary:
                xs, ys = zip(*pts_primary)
                lbl = mk
                if len(xs) == 1:
                    ax.scatter(xs, ys, color=color, marker="*" if mk == "whisper-telugu" else marker,
                               s=150, zorder=5, label=lbl)
                else:
                    ax.plot(xs, ys, color=color, marker=marker, linewidth=2,
                            markersize=8, label=lbl, zorder=4)

            if pts_greedy and mk in ("wav2vec2-xlsr", "mms-300m"):
                xs, ys = zip(*pts_greedy)
                ax.plot(xs, ys, color=color, marker=marker, linewidth=1.2,
                        linestyle="--", markersize=6, alpha=0.5, zorder=3)

        ax.set_xscale("log")
        ax.set_xlabel("Training samples (log scale)", fontsize=13)
        ax.set_ylabel("WER % on FLEURS te_in (lower = better)", fontsize=13)
        ax.set_title("Telugu ASR Learning Curves — FLEURS te_in WER vs Training Samples", fontsize=14)
        ax.legend(fontsize=11, loc="upper right")
        ax.grid(True, which="both", alpha=0.3)
        ax.invert_yaxis() if False else None  # keep normal orientation

        plt.tight_layout()
        fig.savefig(RESULTS_DIR / "learning_curve_live.png", dpi=150)

        clear_output(wait=True)
        display(fig)
        plt.close(fig)

    except Exception as e:
        print(f"[WARN] Plot update failed (training continues): {e}")


print("update_plot() defined.")

In [ ]:
# ── Helper: CTC Training (wav2vec2-xlsr and mms-300m) ─────────────
# Mirrors NB07 training code structure exactly.

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


@dataclass
class DataCollatorCTCWithPadding:
    """Copied from NB07 — pads input_values and labels independently."""
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, padding=self.padding, return_tensors="pt"
        )
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, padding=self.padding, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch


def _make_ctc_compute_metrics(processor):
    def compute_metrics(pred):
        pred_ids  = np.argmax(pred.predictions, axis=-1)
        label_ids = pred.label_ids.copy()
        label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
        pred_str  = processor.batch_decode(pred_ids)
        label_str = processor.batch_decode(label_ids, group_tokens=False)
        return {
            "wer": wer_metric.compute(predictions=pred_str, references=label_str),
            "cer": cer_metric.compute(predictions=pred_str, references=label_str),
        }
    return compute_metrics


def _prepare_ctc_dataset(train_ds, val_ds, processor):
    def prepare(batch):
        audio = batch["audio"]
        batch["input_values"] = processor(
            audio["array"], sampling_rate=audio["sampling_rate"]
        ).input_values[0]
        batch["input_length"] = len(batch["input_values"])
        batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
        return batch

    KEEP = {"input_values", "input_length", "labels"}
    train_prep = train_ds.map(
        prepare,
        remove_columns=[c for c in train_ds.column_names if c not in KEEP],
        num_proc=4, desc="prepare train",
    ).sort("input_length")
    val_prep = val_ds.map(
        prepare,
        remove_columns=[c for c in val_ds.column_names if c not in KEEP],
        num_proc=4, desc="prepare val",
    )
    return train_prep, val_prep


def run_ctc_experiment(
    model_key: str,
    reg: dict,
    train_ds,
    val_ds,
    run_tag: str,
):
    """Train a CTC model (wav2vec2-xlsr or mms-300m adapter).

    Returns (greedy_wer_val, greedy_cer_val, model, processor).
    """
    is_mms = reg.get("adapter", False)

    # Load processor
    if is_mms:
        processor = Wav2Vec2Processor.from_pretrained(reg["model_id"])
    else:
        proc_path = (
            "./telugu_wav2vec2_processor_v2"
            if Path("./telugu_wav2vec2_processor_v2").exists()
            else "./telugu_wav2vec2_processor"
        )
        processor = Wav2Vec2Processor.from_pretrained(proc_path)

    # Prepare features
    train_prep, val_prep = _prepare_ctc_dataset(train_ds, val_ds, processor)
    data_collator = DataCollatorCTCWithPadding(processor=processor)
    compute_metrics = _make_ctc_compute_metrics(processor)

    # Load model
    if is_mms:
        model = Wav2Vec2ForCTC.from_pretrained(
            reg["model_id"],
            ctc_loss_reduction="mean",
            ctc_zero_infinity=True,
            ignore_mismatched_sizes=True,
        )
        model.load_adapter("tel")
        model.freeze_base_model()  # only adapter weights are trainable
    else:
        model = Wav2Vec2ForCTC.from_pretrained(
            reg["model_id"],
            attention_dropout=reg["attention_dropout"],
            hidden_dropout=reg["hidden_dropout"],
            feat_proj_dropout=reg["feat_proj_dropout"],
            mask_time_prob=reg["mask_time_prob"],
            mask_time_length=10,
            mask_feature_prob=reg["mask_feature_prob"],
            ctc_loss_reduction="mean",
            ctc_zero_infinity=True,
            pad_token_id=processor.tokenizer.pad_token_id,
            vocab_size=processor.tokenizer.vocab_size,
            ignore_mismatched_sizes=True,
        )
        model.freeze_feature_encoder()
        model.gradient_checkpointing_enable()

    use_fp16 = (reg["precision"] == "fp16")
    output_dir = f"./wav2vec2-lc-{run_tag}" if not is_mms else f"./mms-lc-{run_tag}"

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=reg["epochs"],
        per_device_train_batch_size=reg["batch"],
        per_device_eval_batch_size=reg["batch"],
        gradient_accumulation_steps=reg["grad_accum"],
        learning_rate=reg["lr"],
        warmup_steps=reg["warmup"],
        lr_scheduler_type="cosine",
        fp16=use_fp16,
        bf16=not use_fp16,
        dataloader_num_workers=4,
        eval_strategy="steps",
        eval_steps=reg["eval_steps"],
        save_strategy="steps",
        save_steps=reg["eval_steps"],
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        logging_steps=100,
        report_to="none",
        seed=SEED,
        remove_unused_columns=False,
        label_names=["labels"],
        push_to_hub=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_prep,
        eval_dataset=val_prep,
        processing_class=processor.feature_extractor,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=reg["patience"],
                early_stopping_threshold=reg["threshold"],
            )
        ],
    )

    print(f"Training CTC [{model_key}] — {len(train_prep):,} samples")
    _ckpts = sorted(Path(output_dir).glob("checkpoint-*"))
    _resume = str(_ckpts[-1]) if _ckpts else None
    if _resume: print(f"  Resuming from {_resume}")
    trainer.train(resume_from_checkpoint=_resume)

    final_dir = os.path.join(output_dir, "final")
    trainer.save_model(final_dir)
    processor.save_pretrained(final_dir)

    val_metrics   = trainer.evaluate()
    greedy_wer    = val_metrics.get("eval_wer", float("nan"))
    greedy_cer    = val_metrics.get("eval_cer", float("nan"))
    epochs_done   = int(trainer.state.epoch or 0)

    print(f"  Greedy WER (val): {greedy_wer:.4f}")
    return greedy_wer, greedy_cer, model, processor, epochs_done


def run_mms_zeroshot(reg: dict):
    """MMS-300M zero-shot: load Telugu adapter, evaluate on FLEURS — no training."""
    processor = Wav2Vec2Processor.from_pretrained(reg["model_id"])
    model = Wav2Vec2ForCTC.from_pretrained(
        reg["model_id"],
        ctc_loss_reduction="mean",
        ctc_zero_infinity=True,
        ignore_mismatched_sizes=True,
    )
    model.load_adapter("tel")
    model = model.eval().cuda()
    return float("nan"), float("nan"), model, processor, 0


print("CTC helpers defined.")

In [ ]:
# ── Helper: Seq2Seq Training (Whisper variants) ────────────────────

import dataclasses


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """Standard HuggingFace Whisper fine-tuning collator."""
    processor: WhisperProcessor

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_features = [
            {"input_features": f["input_features"]} for f in features
        ]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        # Strip decoder BOS token prepended by tokenizer
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch


def _make_seq2seq_compute_metrics(processor):
    def compute_metrics(pred):
        pred_ids = pred.predictions
        label_ids = pred.label_ids.copy()
        label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
        pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
        label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
        return {
            "wer": wer_metric.compute(predictions=pred_str, references=label_str),
            "cer": cer_metric.compute(predictions=pred_str, references=label_str),
        }
    return compute_metrics


def _prepare_seq2seq_dataset(train_ds, val_ds, processor):
    def prepare(batch):
        audio = batch["audio"]
        batch["input_features"] = processor.feature_extractor(
            audio["array"],
            sampling_rate=audio["sampling_rate"],
        ).input_features[0]
        batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
        return batch

    KEEP = {"input_features", "labels"}
    train_prep = train_ds.map(
        prepare,
        remove_columns=[c for c in train_ds.column_names if c not in KEEP],
        num_proc=4, desc="prepare train (whisper)",
    )
    val_prep = val_ds.map(
        prepare,
        remove_columns=[c for c in val_ds.column_names if c not in KEEP],
        num_proc=4, desc="prepare val (whisper)",
    )
    return train_prep, val_prep


def run_seq2seq_experiment(
    model_key: str,
    reg: dict,
    train_ds,
    val_ds,
    run_tag: str,
):
    """Fine-tune a Whisper encoder-decoder model.

    Returns (beam_wer_val, beam_cer_val, model, processor, epochs_done).
    """
    processor = WhisperProcessor.from_pretrained(reg["model_id"], language="te", task="transcribe")
    train_prep, val_prep = _prepare_seq2seq_dataset(train_ds, val_ds, processor)
    data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
    compute_metrics = _make_seq2seq_compute_metrics(processor)

    model = WhisperForConditionalGeneration.from_pretrained(reg["model_id"])

    # Critical settings for Telugu generation — prevents garbage output
    model.config.forced_decoder_ids = None
    model.config.suppress_tokens = []
    model.generation_config.language = "te"
    model.generation_config.task = "transcribe"
    model.generation_config.forced_decoder_ids = None

    use_bf16 = (reg["precision"] == "bf16")
    output_dir = f"./whisper-lc-{run_tag}"

    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=reg["epochs"],
        per_device_train_batch_size=reg["batch"],
        per_device_eval_batch_size=reg["batch"],
        gradient_accumulation_steps=reg["grad_accum"],
        learning_rate=reg["lr"],
        warmup_steps=reg["warmup"],
        lr_scheduler_type="cosine",
        fp16=not use_bf16,
        bf16=use_bf16,
        dataloader_num_workers=4,
        predict_with_generate=True,
        generation_max_length=225,
        eval_strategy="steps",
        eval_steps=reg["eval_steps"],
        save_strategy="steps",
        save_steps=reg["eval_steps"],
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        logging_steps=100,
        report_to="none",
        seed=SEED,
        push_to_hub=False,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_prep,
        eval_dataset=val_prep,
        processing_class=processor.feature_extractor,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=reg["patience"],
                early_stopping_threshold=reg["threshold"],
            )
        ],
    )

    print(f"Training seq2seq [{model_key}] — {len(train_prep):,} samples")
    _ckpts = sorted(Path(output_dir).glob("checkpoint-*"))
    _resume = str(_ckpts[-1]) if _ckpts else None
    if _resume: print(f"  Resuming from {_resume}")
    trainer.train(resume_from_checkpoint=_resume)

    final_dir = os.path.join(output_dir, "final")
    trainer.save_model(final_dir)
    processor.save_pretrained(final_dir)

    val_metrics = trainer.evaluate()
    beam_wer    = val_metrics.get("eval_wer", float("nan"))
    beam_cer    = val_metrics.get("eval_cer", float("nan"))
    epochs_done = int(trainer.state.epoch or 0)

    print(f"  Beam WER (val): {beam_wer:.4f}")
    return beam_wer, beam_cer, model, processor, epochs_done


def eval_whisper_on_fleurs(model, processor, fleurs_test):
    """Run beam-search decoding on FLEURS te_in and return WER."""
    if fleurs_test is None:
        return float("nan")
    device = next(model.parameters()).device
    model.eval()
    preds, refs = [], []
    for i, sample in enumerate(fleurs_test):
        inp = processor.feature_extractor(
            sample["audio"]["array"],
            sampling_rate=16_000,
            return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            generated = model.generate(
                **inp,
                language="te",
                task="transcribe",
                num_beams=5,
            )
        preds.append(processor.tokenizer.decode(generated[0], skip_special_tokens=True))
        refs.append(sample["transcription"])
        if (i + 1) % 200 == 0:
            print(f"  FLEURS eval: {i+1}/{len(fleurs_test)}")
    return wer_metric.compute(predictions=preds, references=refs)


print("Seq2Seq helpers defined.")

In [ ]:
# ── Helper: KenLM Decoding (CTC models only) ───────────────────────
# Copied from NB07 cell-09.

def build_kenlm_arpa(train_ds, run_tag: str) -> Optional[str]:
    """Build a 5-gram ARPA from training transcripts. Returns ARPA path or None."""
    if not Path(KENLM_BIN).exists():
        print(f"[WARN] KenLM binary not found at {KENLM_BIN} — skipping LM decoding.")
        return None

    lm_text_path = f"./telugu_lm_{run_tag}.txt"
    arpa_path    = f"./telugu_kenlm_{run_tag}.arpa"

    with open(lm_text_path, "w", encoding="utf-8") as f:
        f.write("\n".join(train_ds["transcription"]))

    with open(lm_text_path, "rb") as fin, open(arpa_path, "w") as fout:
        subprocess.run(
            [KENLM_BIN, "-o", "5", "--discount_fallback"],
            stdin=fin, stdout=fout, check=True,
        )
    print(f"  ARPA built: {arpa_path}")
    return arpa_path


def decode_kenlm(model, processor, dataset, arpa_path: str) -> float:
    """KenLM beam-search decoding on a dataset. Returns WER."""
    import pyctcdecode

    with open(VOCAB_JSON) as f:
        vocab_dict = json.load(f)
    vocab_list = sorted(vocab_dict.keys(), key=lambda k: vocab_dict[k])
    decoder = pyctcdecode.build_ctcdecoder(vocab_list, arpa_path, alpha=0.5, beta=1.0)

    inference_device = next(model.parameters()).device
    model.eval()
    preds, refs = [], []

    for i, sample in enumerate(dataset):
        inp = processor(
            sample["audio"]["array"],
            sampling_rate=16_000,
            return_tensors="pt",
            padding=True,
        )
        with torch.no_grad():
            logits = model(
                input_values=inp.input_values.to(inference_device)
            ).logits[0].cpu().numpy()
        preds.append(decoder.decode(logits))
        refs.append(sample["transcription"])
        if (i + 1) % 500 == 0:
            print(f"  KenLM decoded: {i+1}/{len(dataset)}")

    return wer_metric.compute(predictions=preds, references=refs)


def decode_greedy_on_fleurs(model, processor, fleurs_test) -> float:
    """Greedy CTC decoding on FLEURS te_in. Returns WER."""
    if fleurs_test is None:
        return float("nan")
    device = next(model.parameters()).device
    model.eval()
    preds, refs = [], []
    for i, sample in enumerate(fleurs_test):
        inp = processor(
            sample["audio"]["array"],
            sampling_rate=16_000,
            return_tensors="pt",
            padding=True,
        )
        with torch.no_grad():
            logits = model(
                input_values=inp.input_values.to(device)
            ).logits
        pred_ids = torch.argmax(logits, dim=-1)
        preds.append(processor.batch_decode(pred_ids)[0])
        refs.append(sample["transcription"])
        if (i + 1) % 200 == 0:
            print(f"  Greedy FLEURS eval: {i+1}/{len(fleurs_test)}")
    return wer_metric.compute(predictions=preds, references=refs)


print("KenLM helpers defined.")

In [ ]:
# ── Helper: Save result ────────────────────────────────────────────

def _safe(v):
    """Round float or return None for NaN."""
    if v is None:
        return None
    try:
        if np.isnan(v):
            return None
        return round(float(v), 4)
    except Exception:
        return None


def save_result(result_dict: dict, model_key: str, source: str, fraction: float):
    """Write per-run JSON and append one line to run_log.txt."""
    frac_tag = f"{fraction:.2f}"
    out_path = RESULTS_DIR / f"{model_key}_lc_{source}_{frac_tag}.json"

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result_dict, f, indent=2)

    log_line = (
        f"{datetime.datetime.now().isoformat()} | "
        f"{result_dict.get('run_tag', '?')} | "
        f"greedy_val={result_dict.get('greedy_wer_val')} | "
        f"kenlm_fleurs={result_dict.get('kenlm_wer_fleurs')} | "
        f"beam_fleurs={result_dict.get('beam_wer_fleurs')}\n"
    )
    with open(RESULTS_DIR / "run_log.txt", "a", encoding="utf-8") as f:
        f.write(log_line)

    print(f"  Saved: {out_path.name}")
    return out_path


print("save_result() defined.")

In [ ]:
# ── MAIN LOOP ──────────────────────────────────────────────────────

experiments = []
for model_key in MODELS_TO_RUN:
    for frac in FRACTIONS[model_key]:
        src = SOURCES[model_key]
        experiments.append((model_key, frac, src))

# Include wav2vec2 full multi-dataset run (pre-populated from NB05)
experiments.append(("wav2vec2-xlsr", 1.00, "multi"))

random.seed(SEED)
random.shuffle(experiments)

print(f"Total experiments: {len(experiments)}")
print(f"Order after shuffle:")
for i, (mk, fr, sc) in enumerate(experiments):
    print(f"  {i+1:2d}. {mk:<20} frac={fr}  source={sc}")

print("\nStarting loop...")

for i, (model_key, frac, src) in enumerate(experiments):
    frac_tag   = f"{frac:.2f}"
    run_tag    = f"{model_key}_{src}_{frac_tag}"
    result_path = RESULTS_DIR / f"{model_key}_lc_{src}_{frac_tag}.json"

    # ── Skip if already done ────────────────────────────────────
    if result_path.exists():
        print(f"[{i+1}/{len(experiments)}] SKIP (already done): {run_tag}")
        continue

    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(experiments)}] START: {run_tag}")
    print(f"{'='*60}")

    reg          = MODEL_REGISTRY[model_key]
    n_train      = 0
    approx_hours = 0.0
    trained_model   = None
    processor_obj   = None
    train_ds_loaded = None

    # ── Load data (zero-shot: no training data needed) ───────────
    if frac == 0.0:
        train_ds_loaded = None
        val_ds_loaded   = None
    else:
        train_ds_loaded, val_ds_loaded, n_train, approx_hours = load_split(src, frac, SEED)

    # ── Train or zero-shot eval ──────────────────────────────────
    greedy_wer_val    = float("nan")
    greedy_cer_val    = float("nan")
    beam_wer_val      = float("nan")
    beam_cer_val      = float("nan")
    kenlm_wer_val     = float("nan")
    kenlm_wer_fleurs  = float("nan")
    greedy_wer_fleurs = float("nan")
    beam_wer_fleurs   = float("nan")
    epochs_done       = 0

    try:
        if reg["arch"] == "ctc":
            if frac == 0.0:
                # MMS zero-shot
                _, _, trained_model, processor_obj, epochs_done = run_mms_zeroshot(reg)
            else:
                greedy_wer_val, greedy_cer_val, trained_model, processor_obj, epochs_done = \
                    run_ctc_experiment(model_key, reg, train_ds_loaded, val_ds_loaded, run_tag)

            # Greedy eval on FLEURS
            print(f"  Evaluating greedy on FLEURS te_in...")
            greedy_wer_fleurs = decode_greedy_on_fleurs(trained_model, processor_obj, fleurs_test)
            print(f"  Greedy WER (FLEURS): {greedy_wer_fleurs:.4f}")

            # KenLM eval
            if reg.get("kenlm") and frac > 0.0:
                arpa_path = build_kenlm_arpa(train_ds_loaded, run_tag)
                if arpa_path:
                    print(f"  Decoding val with KenLM...")
                    kenlm_wer_val = decode_kenlm(
                        trained_model, processor_obj, val_ds_loaded, arpa_path
                    )
                    print(f"  KenLM WER (val): {kenlm_wer_val:.4f}")
                    if fleurs_test is not None:
                        print(f"  Decoding FLEURS te_in with KenLM...")
                        kenlm_wer_fleurs = decode_kenlm(
                            trained_model, processor_obj, fleurs_test, arpa_path
                        )
                        print(f"  KenLM WER (FLEURS): {kenlm_wer_fleurs:.4f}")

        else:  # seq2seq
            beam_wer_val, beam_cer_val, trained_model, processor_obj, epochs_done = \
                run_seq2seq_experiment(model_key, reg, train_ds_loaded, val_ds_loaded, run_tag)

            print(f"  Evaluating beam-search on FLEURS te_in...")
            beam_wer_fleurs = eval_whisper_on_fleurs(trained_model, processor_obj, fleurs_test)
            print(f"  Beam WER (FLEURS): {beam_wer_fleurs:.4f}")

    except Exception as e:
        print(f"[ERROR] Run {run_tag} failed: {e}")
        import traceback; traceback.print_exc()

    # ── Save result (before VRAM cleanup) ────────────────────────
    result = {
        "run_tag":           run_tag,
        "model_key":         model_key,
        "model_id":          reg["model_id"],
        "arch":              reg["arch"],
        "dataset_source":    src,
        "data_fraction":     frac,
        "n_train_samples":   n_train,
        "approx_hours":      approx_hours,
        "greedy_wer_val":    _safe(greedy_wer_val),
        "greedy_cer_val":    _safe(greedy_cer_val),
        "beam_wer_val":      _safe(beam_wer_val),
        "beam_cer_val":      _safe(beam_cer_val),
        "kenlm_wer_val":     _safe(kenlm_wer_val),
        "greedy_wer_fleurs": _safe(greedy_wer_fleurs),
        "kenlm_wer_fleurs":  _safe(kenlm_wer_fleurs),
        "beam_wer_fleurs":   _safe(beam_wer_fleurs),
        "lr":                reg["lr"],
        "effective_batch":   reg["batch"] * reg["grad_accum"],
        "epochs_trained":    epochs_done,
        "timestamp":         datetime.datetime.now().isoformat(),
    }
    save_result(result, model_key, src, frac)

    # ── Live plot update ─────────────────────────────────────────
    update_plot()

    # ── VRAM cleanup (mandatory between runs) ────────────────────
    del trained_model
    del processor_obj
    torch.cuda.empty_cache()
    gc.collect()
    print(f"  VRAM freed. Moving to next run.")

print("\n" + "="*60)
print("All experiments complete.")
print("="*60)
update_plot()

In [ ]:
# ── Final Summary Table ────────────────────────────────────────────
import pandas as pd

rows = []
for p in sorted(RESULTS_DIR.glob("*_lc_*.json")):
    try:
        with open(p) as f:
            rows.append(json.load(f))
    except Exception:
        pass

if not rows:
    print("No results found yet.")
else:
    df = pd.DataFrame(rows)

    # Best FLEURS WER: kenlm > beam > greedy
    def best_fleurs(row):
        for col in ("kenlm_wer_fleurs", "beam_wer_fleurs", "greedy_wer_fleurs"):
            v = row.get(col)
            if v is not None and not pd.isna(v):
                return round(v * 100, 2)
        return None

    df["best_fleurs_wer_%"] = df.apply(best_fleurs, axis=1)

    # Summary view
    summary_cols = [
        "model_key", "data_fraction", "dataset_source",
        "n_train_samples", "approx_hours",
        "greedy_wer_val", "kenlm_wer_val", "beam_wer_val",
        "best_fleurs_wer_%", "epochs_trained", "timestamp",
    ]
    summary_cols = [c for c in summary_cols if c in df.columns]
    display(df[summary_cols].sort_values(["model_key", "data_fraction"]).reset_index(drop=True))

    # Top-5 by FLEURS WER
    print("\n── Top 5 runs by FLEURS WER (lower = better) ──")
    top5 = df.nsmallest(5, "best_fleurs_wer_%")[["model_key", "data_fraction", "dataset_source",
                                                  "n_train_samples", "best_fleurs_wer_%"]]
    display(top5.reset_index(drop=True))

    # Pivot table: model vs fraction
    print("\n── Experiment matrix (FLEURS WER %) ──")
    try:
        pivot = df.pivot_table(
            index="model_key",
            columns="data_fraction",
            values="best_fleurs_wer_%",
            aggfunc="min",
        )
        display(pivot)
    except Exception as e:
        print(f"Pivot failed: {e}")